<a href="https://colab.research.google.com/github/datsay/kg-enhanced-qa-mintaka/blob/main/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets requests tqdm

### Downloading the Mintaka JSON file

In [8]:
# Filter for multihop and intersection questions
filtered_data = [
    item for item in raw_data
    if item.get("complexityType") in ["multihop", "intersection"]
][:100]

print(f"Successfully loaded {len(filtered_data)} evaluation samples.")

processed_samples = []
for item in filtered_data:
    # Extract Wikidata entity IDs and labels from questionEntity
    entities = [
        {"id": e.get("name"), "label": e.get("label")}
        for e in item.get("questionEntity", [])
        if e.get("name")
    ]

    # Extract gold answer strings safely
    gold_answers = []
    ans_obj = item.get("answer") or {}

    if ans_obj.get("mention"):
        gold_answers.append(ans_obj["mention"])

    answer_list = ans_obj.get("answer") or []
    for a in answer_list:
        if isinstance(a, dict) and "label" in a:
            label = a["label"]
            if isinstance(label, dict):
                gold_answers.append(label.get("en"))
            elif isinstance(label, str):
                gold_answers.append(label)
        elif isinstance(a, str):
            gold_answers.append(a)

    # Deduplicate answer strings
    gold_answers = list(set([g for g in gold_answers if g]))

    processed_samples.append({
        "id": item["id"],
        "question": item["question"],
        "complexity": item["complexityType"],
        "entities": entities,
        "gold_answers": gold_answers
    })

first = processed_samples[0]
print("\n--- Example Data Sample ---")
print("ID:", first["id"])
print("Question:", first["question"])
print("Complexity:", first["complexity"])
print("Entities:", first["entities"])
print("Gold Answers:", first["gold_answers"])

Successfully loaded 100 evaluation samples.

--- Example Data Sample ---
ID: fae46b21
Question: What man was a famous American author and also a steamboat pilot on the Mississippi River?
Complexity: intersection
Entities: [{'id': 'Q1497', 'label': 'Mississippi River'}, {'id': 'Q846570', 'label': 'Americans'}]
Gold Answers: ['Mark Twain']
